# 01. Transformers 单卡推理

本 Notebook 调用 `cases/qwen/scripts/run_transformers_torch_npu.py`，不会复制另一份推理实现。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "cases").exists() and (REPO_ROOT.parent / "cases").exists():
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = Path("/mnt/workspace/data/Qwen2.5-0.5B-Instruct")
OUTPUT_DIR = REPO_ROOT / "cases/qwen/results/notebook-inference"

assert (REPO_ROOT / "cases/qwen/scripts/run_transformers_torch_npu.py").exists()
assert (MODEL_PATH / "config.json").exists(), f"找不到模型：{MODEL_PATH}"
print("repo:", REPO_ROOT)
print("model:", MODEL_PATH)

In [ ]:
import torch
import torch_npu
import transformers

print("torch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("transformers:", transformers.__version__)
assert torch.npu.is_available()

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / "cases/qwen/scripts/run_transformers_torch_npu.py"),
    "--model", str(MODEL_PATH),
    "--local-files-only",
    "--prompt", "请用三句话介绍昇腾 CANN。",
    "--max-new-tokens", "32",
    "--warmup", "0",
    "--repeat", "1",
    "--output-dir", str(OUTPUT_DIR),
]
subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
result_file = max(OUTPUT_DIR.glob("transformers_torch_npu_*.json"), key=lambda p: p.stat().st_mtime)
result = json.loads(result_file.read_text(encoding="utf-8"))
for key in ["prompt_tokens", "generated_tokens", "avg_latency_seconds", "tokens_per_second", "memory"]:
    print(f"{key}: {result[key]}")
print("result_file:", result_file)